In [ ]:
# @title 0) Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 1) Install & Import

!pip -q install pycocotools scikit-learn tqdm

import os, json, math, shutil, random, time
import numpy as np
from glob import glob
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
from torchvision.transforms import functional as TVF
from torchvision.ops import box_iou
from torchvision.ops import boxes as box_ops

from PIL import Image
from sklearn.model_selection import train_test_split
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

In [ ]:
# @title 2) Config Path

# --- Drive zips ---
DRIVE_TRAIN_ZIP = '/content/drive/Shareddrives/Workspace/NKla/dataset.zip'
DRIVE_TEST_ZIP  = '/content/drive/Shareddrives/Workspace/NKla/test.zip'

# --- output log dir on Drive ---
EXP_LOG_DIR = '/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments_cascade_frcnn'
os.makedirs(EXP_LOG_DIR, exist_ok=True)

# --- local working dir (fast) ---
LOCAL_ROOT  = '/content/temp_dataset'

# dataset size
TARGET_SIZE = 11000

# class names (11 classes ของมึง)
CLASS_NAMES = [
    'Ascaris lumbricoides', 'Capillaria philippinensis', 'Enterobius vermicularis',
    'Fasciolopsis buski', 'Hookworm egg', 'Hymenolepis diminuta',
    'Hymenolepis nana', 'Opisthorchis viverrine', 'Paragonimus spp',
    'Taenia spp. egg', 'Trichuris trichiura'
]

print("EXP_LOG_DIR =", EXP_LOG_DIR)

In [ ]:
# @title 3) Copy & Unzip to Local temp data

def setup_local_data():
    if os.path.exists(LOCAL_ROOT):
        shutil.rmtree(LOCAL_ROOT)
    os.makedirs(LOCAL_ROOT, exist_ok=True)

    local_train_dir = os.path.join(LOCAL_ROOT, 'train')
    local_test_dir  = os.path.join(LOCAL_ROOT, 'test')
    os.makedirs(local_train_dir, exist_ok=True)
    os.makedirs(local_test_dir,  exist_ok=True)

    print("Copying Train Zip from Drive...")
    shutil.copy(DRIVE_TRAIN_ZIP, '/content/train_temp.zip')
    print("Unzipping Training Data...")
    shutil.unpack_archive('/content/train_temp.zip', local_train_dir)
    os.remove('/content/train_temp.zip')

    if os.path.exists(os.path.join(local_train_dir, 'data')):
        os.rename(os.path.join(local_train_dir, 'data'),
                  os.path.join(local_train_dir, 'images'))

    print("Copying Test Zip from Drive...")
    shutil.copy(DRIVE_TEST_ZIP, '/content/test_temp.zip')
    print("Unzipping Test Data...")
    shutil.unpack_archive('/content/test_temp.zip', local_test_dir)
    os.remove('/content/test_temp.zip')

    if os.path.exists(os.path.join(local_test_dir, 'data')):
        os.rename(os.path.join(local_test_dir, 'data'),
                  os.path.join(local_test_dir, 'images'))

    train_img_dir = os.path.join(local_train_dir, 'images')
    test_img_dir  = os.path.join(local_test_dir, 'images')
    print("DONE! Train images:", len(os.listdir(train_img_dir)))
    print("DONE! Test  images:", len(os.listdir(test_img_dir)))
    return local_train_dir, local_test_dir

LOCAL_TRAIN_DIR, LOCAL_TEST_DIR = setup_local_data()

In [ ]:
# @title 4) Find Coco Prepare Train Val set

def find_any_coco_json(root_dir):
    cands = []
    cands += glob(os.path.join(root_dir, "annotations", "*.json"))
    cands += glob(os.path.join(root_dir, "**", "*.json"), recursive=True)
    for p in cands:
        try:
            with open(p, "r", encoding="utf-8") as f:
                d = json.load(f)
            if isinstance(d, dict) and "images" in d and "annotations" in d and "categories" in d:
                return p
        except:
            pass
    raise FileNotFoundError(f"Cannot find COCO json under: {root_dir}")

TRAIN_IMG_DIR = os.path.join(LOCAL_TRAIN_DIR, "images")
TEST_IMG_DIR  = os.path.join(LOCAL_TEST_DIR,  "images")

TRAIN_JSON_PATH = find_any_coco_json(LOCAL_TRAIN_DIR)
TEST_JSON_PATH  = find_any_coco_json(LOCAL_TEST_DIR)

print("TRAIN_JSON_PATH:", TRAIN_JSON_PATH)
print("TEST_JSON_PATH :", TEST_JSON_PATH)

with open(TRAIN_JSON_PATH, "r", encoding="utf-8") as f:
    coco_full = json.load(f)

images = coco_full["images"][:TARGET_SIZE]
img_ids = [img["id"] for img in images]

train_ids, val_ids = train_test_split(img_ids, test_size=0.2, random_state=42)
train_ids, val_ids = set(train_ids), set(val_ids)

annotations = coco_full["annotations"]
categories  = coco_full["categories"]

images_train = [img for img in images if img["id"] in train_ids]
images_val   = [img for img in images if img["id"] in val_ids]

anns_train = [a for a in annotations if a["image_id"] in train_ids]
anns_val   = [a for a in annotations if a["image_id"] in val_ids]

COCO_TRAIN_JSON = os.path.join(EXP_LOG_DIR, f"coco_train_{TARGET_SIZE}.json")
COCO_VAL_JSON   = os.path.join(EXP_LOG_DIR, f"coco_val_{TARGET_SIZE}.json")

with open(COCO_TRAIN_JSON, "w", encoding="utf-8") as f:
    json.dump({"images": images_train, "annotations": anns_train, "categories": categories}, f)
with open(COCO_VAL_JSON, "w", encoding="utf-8") as f:
    json.dump({"images": images_val, "annotations": anns_val, "categories": categories}, f)

print("Saved:", COCO_TRAIN_JSON)
print("Saved:", COCO_VAL_JSON)
print("Train imgs:", len(images_train), "Val imgs:", len(images_val))

In [ ]:
# @title 5) Dataset & DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

class CocoDetectionForRCNN(torch.utils.data.Dataset):
    def __init__(self, images_dir, ann_json_path, require_anns=False):
        self.images_dir = images_dir
        self.coco = COCO(ann_json_path)

        all_ids = list(sorted(self.coco.imgs.keys()))
        if require_anns:
            self.img_ids = [i for i in all_ids if len(self.coco.getAnnIds(imgIds=[i])) > 0]
        else:
            self.img_ids = all_ids

        cat_ids = list(sorted(self.coco.getCatIds()))
        self.catid_to_contig = {cid: i+1 for i, cid in enumerate(cat_ids)}  # 1..K
        self.contig_to_catid = {v: k for k, v in self.catid_to_contig.items()}

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.images_dir, info["file_name"])

        # เผื่อ extension ไม่ตรง
        if not os.path.exists(img_path):
            base = os.path.splitext(img_path)[0]
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                if os.path.exists(base + ext):
                    img_path = base + ext
                    break

        img = Image.open(img_path).convert("RGB")
        img = TVF.to_tensor(img)  # 0..1

        ann_ids = self.coco.getAnnIds(imgIds=[img_id])
        anns = self.coco.loadAnns(ann_ids)

        boxes, labels, areas, iscrowd = [], [], [], []
        for a in anns:
            x, y, w, h = a["bbox"]
            if w <= 1 or h <= 1:
                continue
            boxes.append([x, y, x+w, y+h])
            labels.append(self.catid_to_contig.get(a["category_id"], 0))
            areas.append(a.get("area", float(w*h)))
            iscrowd.append(a.get("iscrowd", 0))

        if len(boxes) == 0:
            boxes   = torch.zeros((0,4), dtype=torch.float32)
            labels  = torch.zeros((0,),  dtype=torch.int64)
            areas   = torch.zeros((0,),  dtype=torch.float32)
            iscrowd = torch.zeros((0,),  dtype=torch.int64)
        else:
            boxes   = torch.tensor(boxes, dtype=torch.float32)
            labels  = torch.tensor(labels, dtype=torch.int64)
            areas   = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([img_id], dtype=torch.int64),
            "area": areas,
            "iscrowd": iscrowd
        }
        return img, target

train_ds = CocoDetectionForRCNN(TRAIN_IMG_DIR, COCO_TRAIN_JSON, require_anns=True)
val_ds   = CocoDetectionForRCNN(TRAIN_IMG_DIR, COCO_VAL_JSON,   require_anns=True)
test_ds  = CocoDetectionForRCNN(TEST_IMG_DIR,  TEST_JSON_PATH,  require_anns=False)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=2, shuffle=True,
                                           num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=2, shuffle=False,
                                           num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=2, shuffle=False,
                                           num_workers=2, pin_memory=True, collate_fn=collate_fn)

NUM_CLASSES = len(train_ds.catid_to_contig) + 1
print("NUM_CLASSES =", NUM_CLASSES)

In [ ]:
# @title 6) Cascade-ish RoIHeads (3 stages)

# ---------- box encode/decode ----------
def encode_boxes(proposals, gt, stds=(0.1,0.1,0.2,0.2)):
    px1, py1, px2, py2 = proposals.unbind(dim=1)
    gx1, gy1, gx2, gy2 = gt.unbind(dim=1)

    pw = (px2 - px1).clamp(min=1e-6)
    ph = (py2 - py1).clamp(min=1e-6)
    px = px1 + 0.5 * pw
    py = py1 + 0.5 * ph

    gw = (gx2 - gx1).clamp(min=1e-6)
    gh = (gy2 - gy1).clamp(min=1e-6)
    gx = gx1 + 0.5 * gw
    gy = gy1 + 0.5 * gh

    dx = (gx - px) / pw
    dy = (gy - py) / ph
    dw = torch.log(gw / pw)
    dh = torch.log(gh / ph)

    wx, wy, ww, wh = stds
    return torch.stack([dx/wx, dy/wy, dw/ww, dh/wh], dim=1)

def decode_boxes(proposals, deltas, stds=(0.1,0.1,0.2,0.2), clip_val=math.log(1000.0/16)):
    px1, py1, px2, py2 = proposals.unbind(dim=1)
    pw = (px2 - px1).clamp(min=1e-6)
    ph = (py2 - py1).clamp(min=1e-6)
    px = px1 + 0.5 * pw
    py = py1 + 0.5 * ph

    wx, wy, ww, wh = stds
    dx = deltas[:, 0] * wx
    dy = deltas[:, 1] * wy
    dw = (deltas[:, 2] * ww).clamp(max=clip_val)
    dh = (deltas[:, 3] * wh).clamp(max=clip_val)

    gx = dx * pw + px
    gy = dy * ph + py
    gw = torch.exp(dw) * pw
    gh = torch.exp(dh) * ph

    x1 = gx - 0.5 * gw
    y1 = gy - 0.5 * gh
    x2 = gx + 0.5 * gw
    y2 = gy + 0.5 * gh
    return torch.stack([x1, y1, x2, y2], dim=1)

# ---------- heads ----------
class TwoFCHead(nn.Module):
    def __init__(self, in_channels, fc_dim=1024):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, fc_dim)
        self.fc2 = nn.Linear(fc_dim, fc_dim)

    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return x

class ClassAgnosticPredictor(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.cls_score = nn.Linear(in_dim, num_classes)  # include background=0
        self.bbox_pred = nn.Linear(in_dim, 4)            # class-agnostic

    def forward(self, x):
        return self.cls_score(x), self.bbox_pred(x)

def subsample_labels(labels, batch_size=256, pos_fraction=0.25):
    pos_idx = torch.nonzero(labels > 0).squeeze(1)
    neg_idx = torch.nonzero(labels == 0).squeeze(1)

    num_pos = min(int(batch_size * pos_fraction), pos_idx.numel())
    num_neg = min(batch_size - num_pos, neg_idx.numel())

    if pos_idx.numel() > 0:
        pos_idx = pos_idx[torch.randperm(pos_idx.numel(), device=labels.device)[:num_pos]]
    if neg_idx.numel() > 0:
        neg_idx = neg_idx[torch.randperm(neg_idx.numel(), device=labels.device)[:num_neg]]
    keep = torch.cat([pos_idx, neg_idx], dim=0)
    return keep

def fastrcnn_loss_agnostic(class_logits, box_regression, labels, regression_targets, beta=1.0):
    loss_cls = F.cross_entropy(class_logits, labels)
    pos = torch.nonzero(labels > 0).squeeze(1)
    if pos.numel() == 0:
        loss_box = box_regression.sum() * 0.0
    else:
        loss_box = F.smooth_l1_loss(
            box_regression[pos],
            regression_targets[pos],
            beta=beta,
            reduction="sum"
        ) / labels.numel()
    return loss_cls, loss_box

class CascadeRoIHeads(nn.Module):
    def __init__(
        self,
        box_roi_pool,
        num_classes,
        feat_channels,
        num_stages=3,
        stage_iou_thr=(0.5, 0.6, 0.7),
        stage_bbox_stds=((0.1,0.1,0.2,0.2),(0.05,0.05,0.1,0.1),(0.033,0.033,0.067,0.067)),
        stage_loss_weights=(1.0, 0.5, 0.25),
        batch_size_per_image=128,
        positive_fraction=0.25,
        score_thresh=0.05,
        nms_thresh=0.5,
        detections_per_img=100,
        reg_beta=1.0,
        add_gt_as_proposals=True,
    ):
        super().__init__()
        self.box_roi_pool = box_roi_pool
        self.num_classes = num_classes
        self.num_stages = num_stages
        self.stage_iou_thr = stage_iou_thr
        self.stage_bbox_stds = stage_bbox_stds
        self.stage_loss_weights = stage_loss_weights

        self.batch_size_per_image = batch_size_per_image
        self.positive_fraction = positive_fraction

        self.score_thresh = score_thresh
        self.nms_thresh = nms_thresh
        self.detections_per_img = detections_per_img
        self.reg_beta = reg_beta
        self.add_gt_as_proposals = add_gt_as_proposals

        out_size = self.box_roi_pool.output_size
        if isinstance(out_size, int):
            out_size = (out_size, out_size)
        pool_h, pool_w = out_size

        in_dim = feat_channels * pool_h * pool_w

        self.box_heads = nn.ModuleList([TwoFCHead(in_dim, 1024) for _ in range(num_stages)])
        self.box_predictors = nn.ModuleList([ClassAgnosticPredictor(1024, num_classes) for _ in range(num_stages)])

    def has_mask(self):
        return False

    @torch.no_grad()
    def _match_and_label(self, proposals, gt_boxes, gt_labels, thr):
        device = proposals.device
        N = proposals.shape[0]

        if gt_boxes.numel() == 0:
            matched_idx = torch.full((N,), -1, dtype=torch.int64, device=device)
            labels = torch.zeros((N,), dtype=torch.int64, device=device)
            return matched_idx, labels

        ious = box_iou(gt_boxes, proposals)  # [G,N]
        max_iou, argmax = ious.max(dim=0)    # [N]
        matched_idx = argmax
        labels = gt_labels[matched_idx].clone()
        labels[max_iou < thr] = 0
        return matched_idx, labels

    def _refine(self, proposals, deltas, stds, image_shape):
        boxes = decode_boxes(proposals, deltas, stds=stds)
        boxes = box_ops.clip_boxes_to_image(boxes, image_shape)
        return boxes

    def forward(self, features, proposals, image_shapes, targets=None):
        device = proposals[0].device

        if targets is not None:
            losses = {}
            cur_props = proposals

            for s in range(self.num_stages):
                thr  = self.stage_iou_thr[s]
                stds = self.stage_bbox_stds[s]
                w    = self.stage_loss_weights[s]

                sampled_props, sampled_labels, sampled_reg_targets = [], [], []
                per_image_counts = []

                for props_i, tgt_i, img_shape in zip(cur_props, targets, image_shapes):
                    gt_boxes  = tgt_i["boxes"]
                    gt_labels = tgt_i["labels"]

                    if self.add_gt_as_proposals and gt_boxes.numel() > 0:
                        props_i = torch.cat([props_i, gt_boxes], dim=0)

                    matched_idx, labels = self._match_and_label(props_i, gt_boxes, gt_labels, thr=thr)
                    keep = subsample_labels(labels, self.batch_size_per_image, self.positive_fraction)

                    props_keep  = props_i[keep]
                    labels_keep = labels[keep]

                    reg_t = torch.zeros((props_keep.shape[0], 4), dtype=torch.float32, device=device)
                    pos = torch.nonzero(labels_keep > 0).squeeze(1)
                    if pos.numel() > 0 and gt_boxes.numel() > 0:
                        gt_pos = gt_boxes[matched_idx[keep][pos]]
                        reg_t[pos] = encode_boxes(props_keep[pos], gt_pos, stds=stds)

                    sampled_props.append(props_keep)
                    sampled_labels.append(labels_keep)
                    sampled_reg_targets.append(reg_t)
                    per_image_counts.append(props_keep.shape[0])

                box_feats = self.box_roi_pool(features, sampled_props, image_shapes)
                box_feats = self.box_heads[s](box_feats)
                class_logits, box_regression = self.box_predictors[s](box_feats)

                labels_cat = torch.cat(sampled_labels, dim=0)
                reg_targets_cat = torch.cat(sampled_reg_targets, dim=0)

                loss_cls, loss_box = fastrcnn_loss_agnostic(
                    class_logits, box_regression, labels_cat, reg_targets_cat, beta=self.reg_beta
                )
                losses[f"loss_cls_s{s+1}"] = loss_cls * w
                losses[f"loss_box_s{s+1}"] = loss_box * w

                # refine -> next stage proposals
                with torch.no_grad():
                    splits = torch.split(box_regression, per_image_counts, dim=0)
                    next_props = []
                    for props_i, deltas_i, img_shape in zip(sampled_props, splits, image_shapes):
                        refined = self._refine(props_i, deltas_i, stds=stds, image_shape=img_shape)
                        next_props.append(refined.detach())
                    cur_props = next_props

            # empty detections for train mode
            dets = []
            for _ in image_shapes:
                dets.append({
                    "boxes": torch.zeros((0,4), device=device),
                    "labels": torch.zeros((0,), dtype=torch.int64, device=device),
                    "scores": torch.zeros((0,), device=device),
                })
            return dets, losses

        else:
            cur_props = proposals
            last_logits = None

            for s in range(self.num_stages):
                stds = self.stage_bbox_stds[s]

                box_feats = self.box_roi_pool(features, cur_props, image_shapes)
                box_feats = self.box_heads[s](box_feats)
                class_logits, box_regression = self.box_predictors[s](box_feats)

                counts = [p.shape[0] for p in cur_props]
                logits_split = torch.split(class_logits, counts, dim=0)
                deltas_split = torch.split(box_regression, counts, dim=0)

                next_props = []
                for props_i, deltas_i, img_shape in zip(cur_props, deltas_split, image_shapes):
                    refined = self._refine(props_i, deltas_i, stds=stds, image_shape=img_shape)
                    next_props.append(refined)

                cur_props = next_props
                last_logits = logits_split

            # postprocess
            detections = []
            for boxes_i, logits_i, img_shape in zip(cur_props, last_logits, image_shapes):
                boxes_i = box_ops.clip_boxes_to_image(boxes_i, img_shape)
                probs = F.softmax(logits_i, dim=1)

                all_boxes, all_scores, all_labels = [], [], []
                for c in range(1, self.num_classes):
                    sc = probs[:, c]
                    keep = torch.nonzero(sc > self.score_thresh).squeeze(1)
                    if keep.numel() == 0:
                        continue
                    all_boxes.append(boxes_i[keep])
                    all_scores.append(sc[keep])
                    all_labels.append(torch.full((keep.numel(),), c, dtype=torch.int64, device=boxes_i.device))

                if len(all_boxes) == 0:
                    detections.append({
                        "boxes": torch.zeros((0,4), device=boxes_i.device),
                        "scores": torch.zeros((0,), device=boxes_i.device),
                        "labels": torch.zeros((0,), dtype=torch.int64, device=boxes_i.device),
                    })
                    continue

                boxes_cat  = torch.cat(all_boxes,  dim=0)
                scores_cat = torch.cat(all_scores, dim=0)
                labels_cat = torch.cat(all_labels, dim=0)

                keep_small = box_ops.remove_small_boxes(boxes_cat, min_size=1.0)
                boxes_cat  = boxes_cat[keep_small]
                scores_cat = scores_cat[keep_small]
                labels_cat = labels_cat[keep_small]

                keep_nms = box_ops.batched_nms(boxes_cat, scores_cat, labels_cat, self.nms_thresh)
                keep_nms = keep_nms[: self.detections_per_img]

                detections.append({
                    "boxes":  boxes_cat[keep_nms],
                    "scores": scores_cat[keep_nms],
                    "labels": labels_cat[keep_nms],
                })
            return detections, {}

In [ ]:
# @title 7) Build Model (ResNet50-FPN + Replace roi_heads) + OOM tweak

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

MIN_SIZE = 600   # OOM -> 512
MAX_SIZE = 1000  # OOM -> 800

# backbone+RPN pretrained (ช่วยให้เทรนไวขึ้น)
base = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights="DEFAULT",
    min_size=MIN_SIZE,
    max_size=MAX_SIZE
)

# ลด proposals กัน OOM (เหมือนที่มึงทำ)
rpn = base.rpn
if hasattr(rpn, "_pre_nms_top_n") and isinstance(rpn._pre_nms_top_n, dict):
    rpn._pre_nms_top_n["training"] = 1000
    rpn._pre_nms_top_n["testing"]  = 1000
if hasattr(rpn, "_post_nms_top_n") and isinstance(rpn._post_nms_top_n, dict):
    rpn._post_nms_top_n["training"] = 500
    rpn._post_nms_top_n["testing"]  = 500

feat_channels = getattr(base.backbone, "out_channels", 256)

base.roi_heads = CascadeRoIHeads(
    box_roi_pool=base.roi_heads.box_roi_pool,
    num_classes=NUM_CLASSES,
    feat_channels=feat_channels,
    num_stages=3,
    stage_iou_thr=(0.5, 0.6, 0.7),
    stage_bbox_stds=((0.1,0.1,0.2,0.2),(0.05,0.05,0.1,0.1),(0.033,0.033,0.067,0.067)),
    stage_loss_weights=(1.0, 0.5, 0.25),
    batch_size_per_image=128,   # ลดจาก 512 กัน OOM
    positive_fraction=0.25,
    score_thresh=0.05,
    nms_thresh=0.5,
    detections_per_img=100,
)

model = base.to(device)

In [ ]:
# @title 8) Build Model (ResNet50-FPN + Replace roi_heads) + OOM tweak

ACCUM_STEPS = 4

def fix_coco_gt(coco):
    changed = False
    for a in coco.dataset.get("annotations", []):
        if "iscrowd" not in a:
            a["iscrowd"] = 0
            changed = True
        if "area" not in a:
            bb = a.get("bbox", None)
            if bb and len(bb) == 4:
                a["area"] = float(bb[2] * bb[3])
                changed = True
    if changed:
        coco.createIndex()
    return coco

coco_val_gt  = fix_coco_gt(val_ds.coco)
coco_test_gt = fix_coco_gt(test_ds.coco)

@torch.no_grad()
def coco_evaluate(model, data_loader, coco_gt, contig_to_catid, device):
    model.eval()
    coco_results = []

    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = [img.to(device, non_blocking=True) for img in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):
            image_id = int(tgt["image_id"].item())
            boxes  = out["boxes"].detach().cpu().numpy()
            scores = out["scores"].detach().cpu().numpy()
            labels = out["labels"].detach().cpu().numpy()

            for box, score, lab in zip(boxes, scores, labels):
                x1, y1, x2, y2 = box.tolist()
                w, h = x2 - x1, y2 - y1
                if w <= 0 or h <= 0:
                    continue
                coco_results.append({
                    "image_id": image_id,
                    "category_id": int(contig_to_catid.get(int(lab), int(lab))),
                    "bbox": [x1, y1, w, h],
                    "score": float(score),
                })

        torch.cuda.empty_cache() if device.type == "cuda" else None

    if len(coco_results) == 0:
        return {"AP": 0.0, "AP50": 0.0, "AR100": 0.0}

    coco_dt = coco_gt.loadRes(coco_results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    stats = coco_eval.stats
    return {"AP": float(stats[0]), "AP50": float(stats[1]), "AR100": float(stats[8])}

def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    losses = []
    pbar = tqdm(data_loader, desc=f"Epoch {epoch} [train]")

    optimizer.zero_grad(set_to_none=True)

    for step, (images, targets) in enumerate(pbar, 1):
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device.type=="cuda")):
            loss_dict = model(images, targets)
            loss = sum(l for l in loss_dict.values()) / ACCUM_STEPS

        loss.backward()

        if step % ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item() * ACCUM_STEPS)
        pbar.set_postfix(loss=float(np.mean(losses)))

        del images, targets, loss_dict, loss

    if (step % ACCUM_STEPS) != 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if device.type == "cuda":
        torch.cuda.empty_cache()
    return float(np.mean(losses))

In [ ]:
# @title 9) Optimizer + Checkpoint paths (Drive) + Resume option


params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.0025, momentum=0.9, weight_decay=0.0005)

EPOCHS = 10

CKPT_DIR  = os.path.join(EXP_LOG_DIR, "checkpoints_cascade")
os.makedirs(CKPT_DIR, exist_ok=True)

BEST_PATH = os.path.join(CKPT_DIR, "best.pt")
LAST_PATH = os.path.join(CKPT_DIR, "last.pt")
HIST_PATH = os.path.join(EXP_LOG_DIR, f"history_cascade_{TARGET_SIZE}.csv")

def save_ckpt(path, epoch, val_metrics, best_epoch, best_ap):
    torch.save({
        "epoch": epoch,
        "val_metrics": val_metrics,
        "best_epoch": best_epoch,
        "best_AP": best_ap,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "num_classes": NUM_CLASSES,
        "MIN_SIZE": MIN_SIZE,
        "MAX_SIZE": MAX_SIZE,
        "catid_to_contig": train_ds.catid_to_contig,
        "contig_to_catid": train_ds.contig_to_catid,
        "CLASS_NAMES": CLASS_NAMES,
        "cascade_cfg": {
            "stages": 3,
            "iou_thr": (0.5,0.6,0.7),
            "bbox_stds": ((0.1,0.1,0.2,0.2),(0.05,0.05,0.1,0.1),(0.033,0.033,0.067,0.067)),
            "loss_w": (1.0,0.5,0.25),
        }
    }, path)

def try_resume(path=LAST_PATH):
    if not os.path.exists(path):
        return 1, float("-inf"), 0
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = int(ckpt["epoch"]) + 1
    best_ap = float(ckpt.get("best_AP", float("-inf")))
    best_epoch = int(ckpt.get("best_epoch", 0))
    print(f"Resumed from {path} | next_epoch={start_epoch} | best_AP={best_ap:.4f} (epoch {best_epoch})")
    return start_epoch, best_ap, best_epoch

start_epoch, best_ap, best_epoch = try_resume(LAST_PATH)
print("BEST_PATH:", BEST_PATH)
print("LAST_PATH:", LAST_PATH)

In [ ]:
# @title 10) Train Loop + Save best.pt/last.pt

import pandas as pd

history = []
if os.path.exists(HIST_PATH):
    try:
        history = pd.read_csv(HIST_PATH).to_dict("records")
        print("Loaded history:", HIST_PATH, "rows =", len(history))
    except:
        history = []

for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)

    val_metrics = coco_evaluate(model, val_loader, coco_val_gt, train_ds.contig_to_catid, device)
    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)

    pd.DataFrame(history).to_csv(HIST_PATH, index=False)

    save_ckpt(LAST_PATH, epoch, val_metrics, best_epoch, best_ap)

    if val_metrics["AP"] > best_ap:
        best_ap = val_metrics["AP"]
        best_epoch = epoch
        save_ckpt(BEST_PATH, epoch, val_metrics, best_epoch, best_ap)
        print(f"NEW BEST: epoch={epoch} val_AP={best_ap:.4f} -> saved to {BEST_PATH}")

    print("Epoch", epoch, row)

In [ ]:
import torch

def save_model_to_drive(model, save_path):
    try:
        torch.save(model.state_dict(), save_path)
        print(f"✅ Model saved successfully to {save_path}")
    except Exception as e:
        print(f"⚠️ Error saving model: {e}")

model_save_path = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments_cascade_frcnn/cascade_rcnn_best.pt"
save_model_to_drive(model, model_save_path)

In [ ]:
# @title 11) Test Evaluation

test_metrics = coco_evaluate(model, test_loader, coco_test_gt, train_ds.contig_to_catid, device)
print("TEST metrics:", test_metrics)
print("Best checkpoint =", BEST_PATH)

In [ ]:
import time, os, gc, torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from torchvision.ops import box_iou
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def cuda_sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

@torch.no_grad()
def timed_frcnn_predict(frcnn_model, img_t, device):
    img_t = img_t.to(device, non_blocking=True)
    cuda_sync()
    t0 = time.perf_counter()
    out = frcnn_model([img_t])[0]
    cuda_sync()
    t1 = time.perf_counter()
    return out, (t1 - t0)

# Compute Confusion Matrix + mIoU for Cascade R-CNN
def compute_per_class_ious(gt_boxes, gt_cls, pb, pl, ps, conf_thr, iou_thr, K):
    keep = [i for i,s in enumerate(ps) if (s >= conf_thr and pl[i] is not None)]
    pb = pb[keep] if len(keep) else np.zeros((0,4), dtype=np.float32)
    pl = [pl[i] for i in keep]
    ps = [ps[i] for i in keep]

    per_class_vals = [[] for _ in range(K)]
    for c in range(K):
        gt_idx = np.where(np.array(gt_cls) == c)[0]
        if gt_idx.size == 0:
            continue

        vals = [0.0] * int(gt_idx.size)
        pr_idx = [j for j,pc in enumerate(pl) if pc == c]
        if len(pr_idx) > 0:
            gtb = torch.tensor(gt_boxes[gt_idx], dtype=torch.float32)
            prb = torch.tensor(pb[pr_idx], dtype=torch.float32)
            iou_mat = box_iou(prb, gtb)

            used_p=set(); used_g=set()
            flat = iou_mat.flatten()
            order = torch.argsort(flat, descending=True)
            P,G = iou_mat.shape

            for idx_flat in order.tolist():
                p = idx_flat // G
                g = idx_flat %  G
                if p in used_p or g in used_g:
                    continue
                v = float(iou_mat[p,g].item())
                if v < iou_thr:
                    break
                used_p.add(p); used_g.add(g)
                vals[g] = v

        per_class_vals[c].extend(vals)

    return per_class_vals

K = len(CLASS_NAMES)
name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}

def frcnn_label_to_classidx(label_contig, ds):
    lab = int(label_contig)
    if hasattr(ds, "contig_to_catid") and lab in ds.contig_to_catid:
        catid = ds.contig_to_catid[lab]
        nm = ds.coco.cats.get(catid, {}).get("name", None)
        if nm in name_to_idx:
            return name_to_idx[nm]
    x = lab - 1
    return x if 0 <= x < K else None

def load_frcnn_ckpt(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt.get("model_state_dict", ckpt.get("model_state", None))
    if state is None:
        raise KeyError("FRCNN ckpt missing model_state_dict/model_state")
    num_classes = int(ckpt.get("num_classes", K + 1))
    MIN_SIZE = int(ckpt.get("MIN_SIZE", 600))
    MAX_SIZE = int(ckpt.get("MAX_SIZE", 1000))

    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone=None, min_size=MIN_SIZE, max_size=MAX_SIZE
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.load_state_dict(state)
    model.to(device).eval()
    return model

def eval_test_frcnn_miou_and_time(best_ckpt, ds, img_root, conf_thr=0.25, iou_thr=0.5, max_images=None, warmup=10):
    model = load_frcnn_ckpt(best_ckpt, device)

    per_class_all = [[] for _ in range(K)]
    total_inf_time = 0.0
    n = len(ds) if max_images is None else min(len(ds), max_images)

    for i in range(min(warmup, n)):
        img, _ = ds[i]
        _ = model([img.to(device)])[0]

    for i in tqdm(range(n), desc=f"FRCNN test ({os.path.basename(best_ckpt)})"):
        img, tgt = ds[i]
        gt_boxes = tgt["boxes"].numpy()
        gt_labels_raw = tgt["labels"].numpy().tolist()

        mask = []
        gt_cls = []
        for lab in gt_labels_raw:
            ci = frcnn_label_to_classidx(lab, ds)
            mask.append(ci is not None)
            if ci is not None:
                gt_cls.append(ci)
        mask = np.array(mask, dtype=bool)
        gt_boxes = gt_boxes[mask]

        out, dt = timed_frcnn_predict(model, img, device)
        total_inf_time += dt

        pb = out["boxes"].detach().cpu().numpy()
        ps = out["scores"].detach().cpu().numpy().tolist()
        pl_raw = out["labels"].detach().cpu().numpy().tolist()
        pl = [frcnn_label_to_classidx(lab, ds) for lab in pl_raw]

        per_class_vals = compute_per_class_ious(gt_boxes, gt_cls, pb, pl, ps, conf_thr, iou_thr, K)
        for c in range(K):
            per_class_all[c].extend(per_class_vals[c])

    cls_means = [float(np.mean(v)) if len(v) else np.nan for v in per_class_all]
    valid = [m for m in cls_means if not np.isnan(m)]
    miou_macro = float(np.mean(valid)) if len(valid) else 0.0
    all_vals = [x for v in per_class_all for x in v]
    mean_all = float(np.mean(all_vals)) if len(all_vals) else 0.0

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return mean_all, miou_macro, total_inf_time, n

FRCNN_BEST  = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments_frcnn/best_frcnn11000.pt"
CONF_THR = 0.75
IOU_THR  = 0.75
MAX_IMAGES = None

rows = []
m_all, m_macro, tsec, n = eval_test_frcnn_miou_and_time(FRCNN_BEST, test_ds, TEST_IMG_DIR, CONF_THR, IOU_THR, MAX_IMAGES)
rows.append({"model":"cascade_rcnn", "MeanIoU_allGT":m_all, "mIoU_macro":m_macro,
             "test_imgs":n, "infer_total_sec":tsec, "avg_ms_per_img":(tsec/n)*1000, "fps":n/max(tsec,1e-9)})

df = pd.DataFrame(rows).sort_values("infer_total_sec")
print(df)

out_csv = os.path.join(EXP_LOG_DIR, f"compare_cascade_rcnn_TEST_mIoU_time_iou{IOU_THR}_conf{CONF_THR}.csv")
df.to_csv(out_csv, index=False)
print("saved:", out_csv)